# Probar el modelo Gemma-7b-it afinado con LoRA — ejemplos interactivos

Carga el modelo base `google/gemma-7b-it` en 4-bit + los adaptadores LoRA que
guardó `train_gemma.py` (o `train_gemma.ipynb`) en
`/home/jovyan/labs/gemma-7b-it-samsum-lora`, y genera resúmenes para varios
diálogos de ejemplo. Incluye una comparación opcional contra el modelo base
**sin** el fine-tuning, para ver el efecto real del entrenamiento.

Corre esto en el mismo contenedor JupyterHub donde entrenaste (el que levanta
el `docker-compose.yml` de este lab) — así los pesos base de Gemma ya están
en caché y no hace falta volver a autenticarte con Hugging Face salvo que el
caché se haya perdido.

**Requisito:** que `train_gemma.py` (o su versión notebook) ya haya
terminado — vas a ver el mensaje `Adaptadores guardados en: ...` como última
línea de esa corrida.


## 0. Verificar que los adaptadores existan

In [ ]:
import os

ADAPTER_DIR = os.environ.get("OUTPUT_DIR", "/home/jovyan/labs/gemma-7b-it-samsum-lora")

if not os.path.isdir(ADAPTER_DIR) or not os.listdir(ADAPTER_DIR):
    raise SystemExit(
        f"No encuentro adaptadores en {ADAPTER_DIR}. ¿Ya terminó el entrenamiento? "
        "Revisa que haya impreso 'Adaptadores guardados en: ...' como última línea."
    )

print("Adaptadores encontrados en:", ADAPTER_DIR)
print(os.listdir(ADAPTER_DIR))


## 1. Login en Hugging Face (solo si hace falta)

Si entrenaste en esta misma sesión del contenedor, los pesos base ya están en
caché y puedes saltarte esta celda. Si es un contenedor nuevo, corre el login
igual que en `train_gemma.ipynb`.


In [ ]:
import getpass
from huggingface_hub import login

necesita_login = False  # cambia a True si te da un error 401/403 al cargar el modelo más abajo

if necesita_login:
    hf_token = getpass.getpass("Token de Hugging Face (con acceso a google/gemma-7b-it): ")
    login(token=hf_token)
    del hf_token


## 2. Cargar el modelo base (4-bit) + los adaptadores LoRA

In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "google/gemma-7b-it"

print("CUDA disponible:", torch.cuda.is_available())

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    llm_int8_enable_fp32_cpu_offload=True,
)

# El tokenizer se carga del propio directorio de adaptadores (ahí quedó
# guardado al final del entrenamiento, ya con pad_token configurado).
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

print("Memoria GPU ocupada tras cargar el modelo base (GB):",
      round(torch.cuda.memory_allocated() / 1e9, 2) if torch.cuda.is_available() else "N/A")

model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval()
print("Adaptadores LoRA cargados.")


## 3. Funciones auxiliares de generación

In [ ]:
def build_prompt(dialogue):
    messages = [
        {"role": "user", "content": f"Resume el siguiente diálogo en 1-2 frases:\n\n{dialogue}"},
    ]
    # add_generation_prompt=True agrega el "<start_of_turn>model" para que el
    # modelo sepa que le toca generar la respuesta (a diferencia del
    # entrenamiento, donde se incluye también el turno del assistant).
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def resumir(dialogue, max_new_tokens=64):
    prompt = build_prompt(dialogue)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def resumir_base(dialogue, max_new_tokens=64):
    """Igual que resumir(), pero desactivando el LoRA -para comparar contra
    el modelo base sin fine-tuning, sin tener que cargar dos copias del modelo."""
    with model.disable_adapter():
        return resumir(dialogue, max_new_tokens)


## 4. Ejemplos de prueba

Dos diálogos "de juguete" (no vistos en el entrenamiento) y uno al estilo
samsum (mensajes cortos e informales en inglés, como los que sí vio el
modelo durante el fine-tuning).


In [ ]:
ejemplos = [
    "Carlos: ¿Vas a venir a la reunión de las 3pm?\n"
    "Marta: Sí, ya salgo. ¿La sala sigue siendo la 402?\n"
    "Carlos: Sí, misma sala. Nos vemos ahí.",

    "Sofía: Se me quedó el cargador en tu casa ayer, ¿lo tienes ahí?\n"
    "Diego: Sí, lo vi en la mesa de la sala. Te lo llevo mañana a la oficina.\n"
    "Sofía: Perfecto, gracias!",

    "Ana: hey are we still on for the gym at 6?\n"
    "Leo: yeah but running a bit late, more like 6:20\n"
    "Ana: no worries, I'll grab a locker and wait",
]

for i, dialogo in enumerate(ejemplos, start=1):
    print(f"\n{'=' * 70}\nEjemplo {i}\n{'=' * 70}")
    print("Diálogo:\n" + dialogo)
    print("\n>> Resumen (modelo afinado con LoRA):\n" + resumir(dialogo))


## 5. Comparar contra el modelo base (sin fine-tuning)

Corre el mismo diálogo con y sin el adaptador LoRA, para ver si el
fine-tuning realmente cambió el estilo/formato del resumen (más corto, más
parecido al estilo samsum, etc.).


In [ ]:
dialogo_prueba = ejemplos[0]

print("Diálogo:\n" + dialogo_prueba)
print("\n>> Con LoRA (afinado):\n" + resumir(dialogo_prueba))
print("\n>> Sin LoRA (modelo base):\n" + resumir_base(dialogo_prueba))


## 6. Prueba con tu propio diálogo

In [ ]:
mi_dialogo = """Pega aquí tu propio diálogo, con un salto de línea por turno.
Persona A: ...
Persona B: ..."""

print(resumir(mi_dialogo))


## Notas finales

- Si los resúmenes salen vacíos o cortados, sube `max_new_tokens` (por
  defecto 64) en las llamadas a `resumir(...)`.
- `do_sample=False` genera siempre la misma salida para el mismo diálogo
  (útil para comparar); cambia a `do_sample=True, temperature=0.7` en
  `resumir()`/`resumir_base()` si quieres variedad entre corridas.
- Esta misma lógica existe como script plano en `infer_gemma.py`, con un
  flag `--compare_base` que hace automáticamente el paso 5 para una lista
  más larga de ejemplos, sin tener que correr celda por celda.
